<!-- nav -->
[← 4 · What names refer to](04_What_Names_Refer_To.ipynb) · [Home](../../README.md) · [6 · Trees on the wire →](06_Trees_On_The_Wire.ipynb)

# 5 · One framework, three languages

**The problem.** A system's code isn't in one language. The cold-chain logger's firmware is C++, the dashboard is
TypeScript, the pipeline is Python. Tools that work on one language rarely work on another, so every refactoring, check
and generator is written three times.

mbse-programs has one framework for every language: the same trees, traversals, standards and definitions. This case
study does in C++ and TypeScript what the first four did in Python.

In [1]:
import { Ccpp17, Ccpp20, CcppStandard, Definitions as CppDefinitions, Syntax as S } from "@mbse/programs/Ccpp";
import { walk, type SyntaxNode } from "@mbse/programs/Framework/Syntax";
import { Python314, Syntax as P } from "@mbse/programs/Python";
import { ECMAScript2025, TypeScript59, Definitions as TsDefinitions, Syntax as T } from "@mbse/programs/TypeScript";
import { outline } from "./toolkit.ts";

const cpp = Ccpp20.parse(`struct Reading {
    int raw;
};
bool in_range(const Reading &r) {
    return r.raw >= 200 && r.raw <= 800;
}
`) as S.TranslationUnit;
const ts = TypeScript59.parse(`interface Reading {
    celsius: number;
}
export function inRange(r: Reading): boolean {
    return r.celsius >= 2 && r.celsius <= 8;
}
`) as T.Program;

## Step 1: C and C++

Ccpp is C and C++ as one tree language, organized as the standard's grammar is: specifiers, declarators, statements.
The return statement's expression, in outline:

In [2]:
const inRangeCpp = cpp.items[1] as S.FunctionDefinition;
console.log(outline((inRangeCpp.body as S.CompoundStatement).items[0]));

ReturnStatement
  value: BinaryExpression(operator='&&')
    left: BinaryExpression(operator='>=')
      left: MemberExpression(operator='.')
        object: IdExpression
          name: Identifier(spelling='r')
        member: Identifier(spelling='raw')
      right: IntegerLiteral(spelling='200')
    right: BinaryExpression(operator='<=')
      left: MemberExpression(operator='.')
        object: IdExpression
          name: Identifier(spelling='r')
        member: Identifier(spelling='raw')
      right: IntegerLiteral(spelling='800')


## Step 2: TypeScript and JavaScript

TypeScript's tree follows typescript-estree, the tree ESLint reads; JavaScript is the same tree language without the
types. The same expression:

In [3]:
const exported = ts.body[1] as T.ExportNamedDeclaration;
console.log(outline((exported.declaration as T.FunctionDeclaration).body!.body[0]));

ReturnStatement
  argument: LogicalExpression(operator='&&')
    left: BinaryExpression(operator='>=')
      left: MemberExpression
        object: Identifier(name='r')
        property: Identifier(name='celsius')
      right: Literal(raw='2')
    right: BinaryExpression(operator='<=')
      left: MemberExpression
        object: Identifier(name='r')
        property: Identifier(name='celsius')
      right: Literal(raw='8')


## Step 3: the same tools

`walk`, `Parents`, `copy` and `Transformer` are the framework's, so they work alike on every language. Here one function
renames the parameter `r` to `reading` in all three. Each language spells an identifier its own way (`spelling` in Ccpp
and Python, `name` in TypeScript, as typescript-estree does), and that's all that differs:

In [4]:
function rename(tree: SyntaxNode, from: string, to: string): SyntaxNode {
  for (const node of walk(tree)) {
    if ((node instanceof S.Identifier || node instanceof P.Identifier) && node.spelling === from) node.spelling = to;
    if (node instanceof T.Identifier && node.name === from) node.name = to;
  }
  return tree;
}

const py = Python314.parse("def in_range(r):\n    return 200 <= r.raw <= 800\n");
console.log(Ccpp20.print(rename(cpp, "r", "reading")));
console.log(TypeScript59.print(rename(ts, "r", "reading")));
console.log(Python314.print(rename(py, "r", "reading")));

struct Reading {
    int raw;
};
bool in_range(const Reading &reading) {
    return reading.raw >= 200 && reading.raw <= 800;
}



interface Reading {
    celsius: number;
}
export function inRange(reading: Reading): boolean {
    return reading.celsius >= 2 && reading.celsius <= 8;
}



def in_range(reading):
    return 200 <= reading.raw <= 800



(A rename that respects scopes uses each language's `Definitions`, as case study 4 did for Python.)

## Step 4: standards in every family

C and C++ are one tree language whose standards are both families'. C has no references and, before C23, no `bool`;
C++17 has no spaceship operator. TypeScript's own syntax isn't ECMAScript:

In [5]:
console.log(new CcppStandard(2011, "C").check(cpp));
console.log(Ccpp17.check(Ccpp20.parse("auto c = 1 <=> 2;\n")));
console.log(ECMAScript2025.check(ts).slice(0, 2));

[
  "items[1].specifiers[0]: PrimitiveTypeSpecifier.keyword 'bool' needs C23",
  "items[1].declarator.parameters[0].declarator: ReferenceDeclarator is not C"
]


[
  "items[0].declarators[0].initializer.value: BinaryExpression.operator '<=>' needs C++20"
]


[
  "body[0]: TSInterfaceDeclaration is not ECMAScript",
  "body[0].body: TSInterfaceBody is not ECMAScript"
]


## Step 5: definitions in every language

Each language's `Definitions` follows its own rules: C++'s classes and namespaces, TypeScript's values, types and
namespaces kept apart:

In [6]:
console.log([...CppDefinitions.define(cpp).entities()].map(String));
console.log([...TsDefinitions.define(ts).entities()].map(String));

[
  "<class Reading>",
  "<field Reading::raw>",
  "<function in_range>",
  "<parameter in_range::reading>"
]


[
  "<interface Reading>",
  "<property Reading.celsius>",
  "<function inRange>",
  "<parameter inRange.reading>"
]


## Why it works this way

- **A framework, then languages.** Trees, builders, traversals, validation, standards and definitions are written once;
  a language is its kinds, its parser and printer, and its scoping rules. Adding a language adds no new tools.
- **Each language's established tree.** Python's follows `ast`, TypeScript's typescript-estree, C++'s the standard's
  grammar, so each language's programmers recognize theirs.
- **C and C++, TypeScript and JavaScript, as one each.** Their standards overlap so much that one tree language with
  two families of standards says more, and checks more, than two.

<!-- nav -->
[← 4 · What names refer to](04_What_Names_Refer_To.ipynb) · [Home](../../README.md) · [6 · Trees on the wire →](06_Trees_On_The_Wire.ipynb)